In [ ]:
import numpy as np
import time

import mbuild as mb
from mbuild import Polymer
from mbuild.path import Path, cyclic, lamellar, hard_sphere_random_walk

from mbuild.exceptions import PathConvergenceError

from mbuild.simulation import HoomdSimulation, hoomd_fire, hoomd_cap_displacement, ForcesHandler

from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

gaff = ForceField("../files/gaff.xml")
oplsaa = ForceField("oplsaa")
spcfw = ForceField("../files/spcfw.xml")

# Relax Compound:
- Can put the HOOMD-Blue based relaxation steps into a function
- You can pass an mBuild Compound into this method to run the cap displacement + FIRE simulation steps.
- Change parameters like number of steps, number of FIRE iterations and forcefield

In [ ]:
def relax_compound(compound, cap_displacement_steps=1000, fire_iterations=5, fire_steps=1000, forcefield=gaff):
    """"""
    hoomd_sim = HoomdSimulation(compound=compound, forcefield=forcefield, r_cut=1.2)
    hoomd_cap_displacement(
        n_steps=cap_displacement_steps,
        compound=compound,
        sim=hoomd_sim,
        forces_handler=ForcesHandler(),
        max_displacement=1e-3,
        dt=1
    )
    hoomd_fire(compound=compound, sim=hoomd_sim, n_iterations=5, n_steps=500, forces_handler=ForcesHandler())

# Path Module: External Library
- Several external classes are usable with `hard_sphere_random_walk`
- This includes:
    - `Constraint`: Volume constraints with settable periodic boundary conditions.
    - `Termination`: Triggers that end a walk (more than just N steps).
    - `Bias`: Methods that influence next-steps towards some conditional.
    - `AnglesSampler`: Provides different options to determining how angles are sampled during random walks.
    - `BeadNamer`: Rules for naming each step of the random walk.
- The imports below illustrate the different types of each currently available:

In [ ]:
from mbuild.path.constraints import CuboidConstraint, SphereConstraint, CylinderConstraint
from mbuild.path.termination import Termination, NumSites, NumAttempts, WithinCoordinate, RadiusOfGyration
from mbuild.path.bias import TargetType, TargetDirection, AvoidType, AvoidDirection, TargetCoordinate, AvoidCoordinate
from mbuild.path.points import AnglesSampler
from mbuild.path.namers import RandomNamer, CyclicNamer, MarkovNamer, ConstantNamer

#### Tip:
- Trying runnin `help(class)` on any of the above imports to see doc strings and information about each.
- Replace `class` with the specific class you want to view

# Running multiple random walks:
- The `hard_sphere_random_walk` method can be called in a for loop to create multichain systems.
- Each random walk know about the coordinates of all previous random walks
- This is achieved by passing in a `Path()` object
- Below: Runs 50 random walks of 50 sites each.

In [ ]:
num_chains = 50
chain_lengths = 50

system = Path()
for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        termination=chain_lengths,
        radius=1,
        seed=42,
        bond_length=1.01
    )
system.visualize(radius=1)

# Controlling chain angles
- There are multiple ways to influence the chain configurations you get from a random walk
- Below are multiple options of specifying the angle sampling to use.
- These can include:
    - Passing a tupe of `(min, max)` which samples uniformly between the min and max angle
    - Creating an `AnglesSampler` object, specifying the type of sampling (`uniform`, `normal`, `choice`)
- This class is extensible:
    - You can inherit from `AnglesSampler` to implement your own method of determining the angles used during each random walk step.
    - This can be designed externally to random walk, and passed in without changing the underlying code.

## Below:
- Try uncommenting (delete the hashtag) for the different `sample_angles` instances.
- Run the cell and observe the impact it has on chain conformations

In [ ]:
num_chains = 50
chain_lengths = 50

sample_angles = (np.pi/2, np.pi/1.8) # Sampling uniformly from a small range of small angles

#sample_angles = (np.pi/1.2, np.pi) # Sampling uniformly from a small range of large angles

#sample_angles = AnglesSampler("normal", dict(loc=1.6, scale=0.3)) # Sampling from normal distribution, small average

#sample_angles = AnglesSampler("normal", dict(loc=2.6, scale=0.3)) # Sampling from normal distribution, large average

#sample_angles = AnglesSampler("normal", dict(loc=2.3, scale=0.8)) # Sampling from normal distribution, larger std

#sample_angles = AnglesSampler("choice", dict(a=np.pi/2)) # Every angle is 90 degrees

#sample_angles = AnglesSampler("choice", dict(a=[np.pi/2, np.pi], p=[0.50, 0.50])) # ~50/50 split between 90deg and 180deg

system = Path()
for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        termination=chain_lengths,
        radius=1,
        seed=42,
        bond_length=1.01,
        rw_angles=sample_angles # Change the (min, max) angles to change chain conformations
    )
system.visualize(radius=1)

# Volume Constraints
- Volume constraints can be used during random walks.
- These also exist externally to the random walk core logic.
- This means you can make your own `Constraint` class that implements your own volume constraints.
- Similar to `AnglesSampler`, this is defined separately, and passed into the `hard_sphere_random_walk` methods.
- Below: We create 3 different volume constraints
    - cuboid
    - sphere
    - cylinder
- **Try:** Pass each one into the `volume_constraint` parameter and see the resulting system. 

In [ ]:
num_chains = 80
chain_lengths = 50

cuboid = CuboidConstraint(Lx=13, Ly=13, Lz=13, pbc=(False, False, False))
sphere = SphereConstraint(radius=8, center=(0,0,0))
cylinder = CylinderConstraint(radius=5, height=18)

system = Path()

for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        volume_constraint=cuboid,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
        seed=45,
        radius=0.34,
        bond_length=0.341
    )

system.visualize(radius=0.34)

# Multiple chain types
- Using for-loops to create multi-chain systems lets you change some parameter/information about each iteration.
- Here, we create a multi-component system where the bead-name changes between `A` and `B`.

In [ ]:
num_chains = 80
chain_lengths = 50

cuboid = CuboidConstraint(Lx=13, Ly=13, Lz=13, pbc=(False, False, False))
sphere = SphereConstraint(radius=8, center=(0,0,0))
cylinder = CylinderConstraint(radius=5, height=18)

system = Path()

for i in range(num_chains):
    if i % 2 == 0:
        name = "A"
    else:
        name = "B"
    rw = hard_sphere_random_walk(
        path=system,
        bead_name=name,
        volume_constraint=sphere,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
        seed=45,
        radius=0.34,
        bond_length=0.341
    )

system.visualize(radius=0.34)

# Packing around an mBuild Compound
- You can pass in an `mBuild.Compound` instane to `hard_sphere_random_walk`.
- This lets you do things such as:
    - Run an random walk between surfaces, around a box of nanotubes, within a nanotube, around a box of solvents
- Below: Use the `mbuild.fill_box()` method to create a low-density system of water molecules
    - Pass this compound into `hard_sphere_random_walk`  

In [ ]:
L = 8
water_box = mb.fill_box(compound=mb.load("O", smiles=True), n_compounds=[200], box=[L, L, L])
water_box.visualize()

In [ ]:
cuboid = CuboidConstraint(Lx=L, Ly=L, Lz=L, center=water_box.center)

num_chains = 25
chain_lengths = 30
system = Path()

for i in range(num_chains):
    rw = hard_sphere_random_walk(
        path=system,
        include_compound=water_box,
        volume_constraint=cuboid,
        termination=chain_lengths,
        rw_angles=AnglesSampler("normal", dict(loc=2.5, scale=0.5)),
        seed=45,
        radius=0.34,
        bond_length=0.341
    )

# Convert the random walk path to a compound, so we can visualize it with the water molecules
water_box.add(system.to_compound())
water_box.visualize(bead_size=0.8)

# Back mapping and relaxing
- Each random walk can be passed through the polymer builder (as shown in the last tutorial).
- This lets you perform back-mapping and relaxing on a chain-by-chain basis.
- Below: Each random walk is back mapped to an atomistic configuration, relaxed, and added to a `system` compound.
    - As the system is updated, it is passed into the next random walk 

In [ ]:
num_chains = 20
chain_lengths = 20

cuboid = CuboidConstraint(Lx=8, Ly=8, Lz=8, pbc=(False, False, False))
sphere = SphereConstraint(radius=8, center=(0,0,0))
cylinder = CylinderConstraint(radius=5, height=18)

system = mb.Compound()

for i in range(num_chains):
    rw = hard_sphere_random_walk(
        include_compound=system,
        volume_constraint=cuboid,
        termination=chain_lengths,
        rw_angles=(1.5, 3.14),
        seed=45+i,
        radius=0.34,
        bond_length=0.341
    )
    peg_monomer = mb.load("C{<}CO{>}", smiles=True)
    peg_chain = Polymer()
    peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
    peg_chain.build_from_path(rw)
    relax_compound(peg_chain, cap_displacement_steps=1000, fire_iterations=5, fire_steps=500, forcefield=gaff)
    system.add(peg_chain)

system.visualize()

# Random Walks From Paths
- Random walks can originate from other paths.
    - This includes running random walk chains branching off previous chains
    - Or, you can create another path object (lamellar, cyclic) and run branching random walks
- Notes:
    - Set `connectivity` to `"link-linear"` to form a bond between the path site the RW is originating from and the first point of the RW. 

In [ ]:
# Create a path object from hard_sphere_random_walk
backbone = hard_sphere_random_walk(termination=50, radius=0.34, bond_length=0.341, rw_angles=(2.8, 3.14))

for starting_index in np.arange(5, 45, 5):
    branch = hard_sphere_random_walk(
        path=backbone,
        termination=20,
        bead_name="B",
        radius=0.34,
        bond_length=0.341,
        initial_point=int(starting_index),
        connectivity="link-linear",
        rw_angles=(2.4, 3.14)
    )

backbone.visualize(radius=0.34)

In [ ]:
ring = cyclic(spacing=0.34, N=50, bead_name="R")

for starting_index in np.arange(0, 50, 10):
    rw = hard_sphere_random_walk(
        path=ring,
        bead_name="A",
        radius=0.34,
        seed=int(starting_index),
        bond_length=0.341,
        initial_point=int(starting_index),
        termination=35,
        connectivity="link-linear",
        rw_angles=(2.5, 3.14)
    )

ring.visualize(radius=0.34)

## Using a Bias
- Biases are also designed externally to the random walk logic.
- This also allows you to build your own implementation of a bias without touching the random walk code.
- Below:
    - Let's create the same system as above, but bias the random walks to run normal to the cyclic path's plane.
- Biases can be assigned a weight that changes how strongly they influence the random walk.
- Try changing `weight` to any number between 0 and 1 

In [ ]:
ring = cyclic(spacing=0.34, N=50, bead_name="R")
bias = TargetDirection(direction=(0, 0, 1), weight=0.75)

for starting_index in np.arange(0, 50, 5):
    rw = hard_sphere_random_walk(
        path=ring,
        bias=bias,
        bead_name="A",
        radius=0.34,
        seed=int(starting_index),
        bond_length=0.341,
        initial_point=int(starting_index),
        termination=35,
        connectivity="link-linear",
        rw_angles=(2.5, 3.14)
    )

ring.visualize(radius=0.34)

## Different Bias
- Below: Use the `AvoidCoordinate` bias to influence random walks to run away from the center of the ring

In [ ]:
ring = cyclic(spacing=0.34, N=50, bead_name="R")
bias = AvoidCoordinate(avoid_coordinate=(0,0,0), weight=0.75)

for starting_index in np.arange(0, 50, 5):
    rw = hard_sphere_random_walk(
        path=ring,
        bias=bias,
        bead_name="A",
        radius=0.34,
        seed=int(starting_index),
        bond_length=0.341,
        initial_point=int(starting_index),
        termination=35,
        connectivity="link-linear",
        rw_angles=(2.5, 3.14)
    )

ring.visualize(radius=0.34)

<h1 style="color: green;">Exercise</h1>

- Creating a lamellar path (2D or 3D)
- First, run a walk from the last index of this path
    - `initial_point = int(len(lamellar_path.coordinates) - 1`)
- Second, run a walk from index zero of this path 
- Remember: Pass in the lamellar path object you create into the `hard_sphere_random_walk` calls.
- Remember: To connect the RW path to a path it is originating from, use `connectivity="link-linear"`  

In [ ]:
# Your code here

<h2 style="color: blue;">Answer</h2>
Click on the cell below to see the answer

In [ ]:
lamellar_2D = lamellar(
    spacing=0.45,
    layer_length=5,
    layer_separation=1.5,
    num_layers=5
)

rw = hard_sphere_random_walk(
    path=lamellar_2D,
    initial_point=int(len(lamellar_2D.coordinates) - 1),
    radius=0.45,
    bond_length=0.451,
    termination=40
)

rw = hard_sphere_random_walk(
    path=lamellar_2D,
    initial_point=0,
    radius=0.45,
    bond_length=0.451,
    termination=40
)

rw.visualize(radius=0.45)

# Termination:
- So far, we have just been passing an integer into the `termination` parameter of `hard_sphere_random_walk`
    - This creates a `NumSites(N)` instance for you.
- Instead you can use the `Termination` classes
    - `termination = NumSites(20)
- Termination conditions are composable: `Termination([list of termination triggers])`
  
    - `termination_conditions = Termination([NumSites(100), NumAttempts(500)])`
        - This terminates once either of these are met

<h1 style="color: green;">Exercise</h1>

- Run a single random walk that terminates once it reaches a target radius of gyration, rather than a specific site count.
- Use the `RadiusofGyration` termination condition rather than `NumSites`
- Note, this is computed as $R_g^2$
- Aim for a $R_g^2 = 100$ 

In [ ]:
from mbuild.path.termination import RadiusOfGyration

In [ ]:
# Your code here

<h2 style="color: blue;">Answer</h2>
Click on the cell below to see the answer

In [ ]:
rw = hard_sphere_random_walk(
    radius=0.40,
    bond_length=0.401,
    termination=RadiusOfGyration(radius_of_gyration=100),
)

rw.visualize(radius=0.40)

### Check the $R_g^2$

NOTE: If you didn't run the "answer" cell above, you might need to change `rw` to whatever variable you assigned in your own code block

In [ ]:
center = rw.coordinates.mean(axis=0)
diffs = rw.coordinates - center
rg2 = np.mean(np.sum(diffs * diffs, axis=1))
print(rg2)

---

# Dense Random Walks: API Under Construction

A couple of notes about running dense random walks:
- Most of the examples above are low density systems, this is partly to make illustration of the API faster and easier.
- Also, this is partly because (at the moment) restart conditions for failed random walks are left up to the users
- **Still in progress:** We are still deciding what how to implement higher-level functions to build to make running dense random walks doable from one function call.
  

Below is one example of adding a `while` loop to the `for` loop steps we've been using so far.
- This uses the `sample_candidates()` method from `Constraint` classes.
- This method uniformly samples the volume constraint, and given a set of obstacle coordinates (previous random walks), sorts this sampled space from lowest to high local density.
- If that initial starting point results in a failed random walk, a new one is sampled.

In [ ]:
n_chains = 100
chain_lengths = 60
L = 6
box = CuboidConstraint(Lx=L, Ly=L, Lz=L, pbc=(True, True, True))
path = Path()

for i in range(n_chains):
    walk_passed = False
    count = 0
    while not walk_passed:
        try:
            term = Termination([NumSites(chain_lengths), NumAttempts(chain_lengths*2)])
            init_points = box.sample_candidates(points=path.coordinates, n_candidates=200+count, buffer=0.01)
            random_walk = hard_sphere_random_walk(
                path=path,
                radius=0.25,
                volume_constraint=box,
                bond_length=0.251,
                initial_point=init_points[count],
                seed=20,
                termination=term,
                rw_angles=AnglesSampler("uniform", dict(low=1.5, high=np.pi)),
                trial_batch_size=30
            )
            if term.success:
                print("Finished", i+1)
                walk_passed = True
            else:
                count += 1
        except PathConvergenceError:
            count += 1

#### Below we can convert this system of random walks to an mBuild Compound so we can utilize the option to hide periodic bonds (makes it easier to see the system)

In [ ]:
compound = path.to_compound()
compound.box = mb.Box([L, L, L])
compound.visualize(bead_size=0.7, periodic_bond_opacity=0.1)

# Another Example: Dense mBuild Compound system
- Goals: Show an example of how you might build an initial semi-crystalline system
- Steps:
    - Create a lamellar path, use the polymer builder to back map and relax
    - Use `mb.fill_box` to randomly place 2 of these lamellar chunks in a box
    - Pass this Compound into the random walk for loop
    - Each random walk is back mapped, and added to the lamellar system

In [ ]:
lamellar_3D = lamellar(
    spacing=0.35,
    layer_length=3,
    layer_separation=0.75,
    num_layers=5,
    stack_separation=0.75,
    num_stacks=4
)
peg_monomer = mb.load("C{<}CO{>}", smiles=True)
peg_chain = Polymer()
peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
peg_chain.build_from_path(path=lamellar_3D)
relax_compound(peg_chain)

peg_chain.visualize()

In [ ]:
peg_box = mb.fill_box(compound=peg_chain, n_compounds=2, box=[10,10,10], seed=23, overlap=1)
peg_box.translate_to((0,0,0)) # Center the system at 0, 0, 0
peg_box.visualize()

### NOTE: This cell below might take 5ish minutes to run

In [ ]:
n_chains = 100
chain_lengths = 60
L = 10
box = CuboidConstraint(Lx=L, Ly=L, Lz=L, pbc=(True, True, True))

for i in range(n_chains):
    walk_passed = False
    count = 0
    while not walk_passed:
        try:
            term = Termination([NumSites(chain_lengths), NumAttempts(chain_lengths*2)])
            init_points = box.sample_candidates(points=peg_box.xyz, n_candidates=200+count, buffer=0.01)
            random_walk = hard_sphere_random_walk(
                include_compound=peg_box,
                radius=0.35,
                volume_constraint=box,
                bond_length=0.351,
                initial_point=init_points[count],
                seed=i+count,
                termination=term,
                rw_angles=AnglesSampler("uniform", dict(low=1.5, high=np.pi)),
                trial_batch_size=100
            )
            if term.success:
                print("Finished", i+1)
                peg_monomer = mb.load("C{<}CO{>}", smiles=True)
                peg_chain = Polymer()
                peg_chain.add_monomer(peg_monomer, head_tag=">", tail_tag="<", separation=0.145)
                peg_chain.build_from_path(path=random_walk)
                peg_box.add(peg_chain)
                walk_passed = True
                
            else:
                count += 1
        except PathConvergenceError:
            count += 1

# Visualize the resulting system.
- Rotate the system around and you can see the randomly oriented crystalline chunks within the amorphous polymers

In [ ]:
peg_box.visualize(periodic_bond_opacity=0.2)

# Hybrid Packing

- All the tools we've talked about can be used to design your own, hybrid packing methods
- Example: MC packing + periodic MD relaxation
    - In the example above, we can aim for more dense packing of a semi-crystalline system by periodically passing the entire system (lamellar + amorphous chains) through a relaxation protocol that includes NVT runs. Then further packing can resume.
- Example: Use a smaller radius to more aggressively allow overlapping particles and run capped displacement with a softer push off potential.

---